# FMSE Laboratorio 06 - Banco de trabajo de estudios de arquitectura

**Curso:** FMSE 2026 · **Módulo 6:** Patrones de arquitectura y estudios de alternativas · **Fase:** Arquitectura · **Nivel:** Ingeniería

La página del curso y tu progreso están en [agentic-ai.es/academy/fmse/learn/m06-architecture-trade-studies](https://agentic-ai.es/academy/fmse/learn/m06-architecture-trade-studies).

> **Sobre el idioma.** El código, las claves de los datos y el escenario de ejemplo se quedan en inglés; los mensajes de los validadores están en español. Los `revisit_triggers` del ADR puedes escribirlos en español (por ejemplo, "si el costo por incidencia supera los 0,05 EUR").

## 0. Misión y competencia objetivo

**Misión.** Diseñar tres arquitecturas para un flujo de soporte de gran volumen con restricciones estrictas de seguridad y de costo, y elegir una solo después de comparar las alternativas con datos medidos o estimados.

**Laboratorio guiado.** Completas matrices de candidatas y gráficos de sensibilidad a partir de datos de benchmark que ya vienen dados.

**Competencia objetivo (resultados del Módulo 6):**
- Comparar los patrones de una sola llamada, pipeline determinista, cadena de prompts, flujo de trabajo u orquestador, agente con herramientas y multiagente.
- Hacer estudios de alternativas, con matriz de Pugh o ponderados, con su análisis de sensibilidad.
- Identificar los límites de confianza, los límites deterministas y las zonas de contención de fallos.
- Redactar registros de decisión de arquitectura (ADR) ligados a los requisitos.

**Guía:** alrededor de un 25% guiado y un 75% por tu cuenta. Es un **reto de diseño**: el experimento guiado (secciones 4 y 5) te da un camino de referencia que funciona; el reto (sección 6), no.

## 1. Requisitos y criterios de aceptación

Estos requisitos los comprueban validadores públicos. Te dicen *qué requisito* falló y por qué, nunca la implementación que se esperaba.

| ID | Requisito |
| --- | --- |
| ARC-01 | >=3 alternativas creíbles |
| ARC-02 | Los criterios se trazan hasta los requisitos |
| ARC-03 | Los pesos suman 1 |
| ARC-04 | Se ha hecho el análisis de sensibilidad |
| ARC-05 | La decisión incluye condiciones para revertirla o revisarla |

**Control de competencia (portal):** la defensa de la arquitectura PASA. También hay que aprobar el cuestionario (>= 80%) y completar el laboratorio guiado, el artefacto y la reflexión. Un requisito crítico no se compensa con otras puntuaciones.

## 2. Preparación del entorno

In [ ]:
# Idempotente: se puede volver a ejecutar y nunca imprime secretos.
LAB_ID = "lab-06"
LABKIT_VERSION = "1.1.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit ya estaba cargado"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"usando el labkit de {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "usando el labkit descargado anteriormente"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"labkit descargado y verificado desde {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "No se pudo cargar fmse_labkit desde: " + "; ".join(problems) + ". "
        "Solución: descarga la carpeta de laboratorios desde agentic-ai.es/academy/fmse/resources, sube la carpeta fmse_labkit junto a este notebook (panel Archivos) y vuelve a ejecutar esta celda."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("es")
print(f"fmse_labkit {fmse.__version__} listo para {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Comprobación de secretos

Este laboratorio funciona por completo sin conexión: no necesita ninguna clave de proveedor. Si usas claves en otros laboratorios, guárdalas en los Secretos de Colab; nunca se imprimen.

In [ ]:
fmse.secrets_check()

## 4. Experimento guiado

Ya tienes los datos de benchmark, estimados, de seis patrones de arquitectura en un flujo de soporte de gran volumen. Las funciones `weighted_totals`, `winner` y `perturb` aplican las convenciones del curso para los estudios de alternativas.

In [ ]:
from fmse_labkit.labs import lab06
import json

guided = fmse.GuidedLog(LAB_ID, required_steps=["matrix_completed", "sensitivity_run"])
print(json.dumps(lab06.SCENARIO, indent=2))
for p, row in lab06.BENCHMARK.items():
    print(f"{p:<24} {row}")

## 5. Predice antes de ejecutar

¿Qué patrón va a ganar en una matriz que da más peso a la seguridad y al costo? ¿Gana el patrón más exacto? Escribe tu predicción y después ejecuta el experimento.

In [ ]:
my_prediction = ""
guided.predict("prediction", my_prediction)

In [ ]:
criteria = [{"id": "sec", "weight": 0.35}, {"id": "cost", "weight": 0.35}, {"id": "acc", "weight": 0.30}]
scores = {"pipeline": {"sec": 5, "cost": 5, "acc": 3}, "orchestrator": {"sec": 4, "cost": 4, "acc": 4}, "agent": {"sec": 2, "cost": 1, "acc": 5}}
print("totales ponderados:", lab06.weighted_totals(scores, criteria), "ganador:", lab06.winner(scores, criteria))
guided.step("matrix_completed")
for cid in ("sec", "cost"):
    for d in (0.2, -0.2):
        print(cid, f"{d:+.0%}", "->", lab06.winner(scores, lab06.perturb(criteria, cid, d)))
guided.step("sensitivity_run")
guided.outcome("prediction", "ver la salida de arriba")

**Hoja de trabajo.** Anota lo que observaste, al menos una frase completa por campo: estas observaciones forman parte de la evidencia de tu laboratorio guiado.

In [ ]:
worksheet = {
    "matrix_observation": "",  # ¿Qué alternativa ganó con los pesos iniciales y por qué?
    "sensitivity_observation": "",  # ¿Cambió el ganador al alterar los pesos? ¿Con qué cambio?
    "prediction_check": "",  # ¿Acertaste en tu predicción? ¿Qué se te escapó?
}
guided.observe(worksheet)

## 6. Reto

**Tres arquitecturas para un flujo de soporte de gran volumen.** Diseña tres arquitecturas para un flujo de soporte de gran volumen con restricciones estrictas de seguridad y de costo. Elige una solo después de comparar las alternativas con datos medidos o estimados.

Restricciones:
- Los criterios tienen que trazarse hasta los ids de los requisitos del escenario.
- Sensibilidad: altera los dos pesos más altos en +/-20% y vuelve a normalizar.

In [ ]:
my_study = {
    "alternatives": [],  # {"id", "name", "pattern" (ver lab06.PATTERNS), "description"}
    "criteria": [],  # {"id", "name", "weight", "requirements": [ids de los requisitos del escenario]}
    "scores": {},  # id de la alternativa -> {id del criterio: 1-5}
    "sensitivity": [],  # {"criterion", "delta": +0.2/-0.2, "winner"} para los dos pesos más altos
    "decision": {"selected": "", "context": "", "rationale": "", "consequences": "", "revisit_triggers": []},
}

## 7. Validación pública

In [ ]:
submission = my_study
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustez / pruebas adversariales

Condiciones adversariales que van más allá de la batería que se califica. No cambian tu puntuación: te muestran dónde es frágil tu solución.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Reflexión de ingeniería

La reflexión que se califica se escribe en el portal y se guarda con tu artefacto. Si te sirve, haz aquí un borrador:
- **Defensa de la arquitectura:** ¿por qué existe la arquitectura que elegiste y por qué descartaste las demás? Apóyate en tu estudio de alternativas como evidencia.
- **Compromisos:** ¿a qué renuncia tu elección y qué cambios en los requisitos te harían dar marcha atrás?
- **Riesgos pendientes:** ¿qué límite de confianza o zona de contención de fallos te preocupa más?

In [ ]:
reflection_draft = {"architecture_defense": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Exportar el artefacto / resultado de finalización

Exporta el artefacto **Especificación de la arquitectura, estudio de alternativas (Pugh) y ADR** e imprime un registro de finalización. Pega el registro en la página del Módulo 6 de agentic-ai.es (**Importar el resultado del laboratorio**). El registro es un documento del flujo de aprendizaje, no un certificado firmado.

In [ ]:
artifact = fmse.export_artifact(LAB_ID, my_study, title="Especificación de la arquitectura, estudio de alternativas (Pugh) y ADR", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, **{}},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)